In [0]:
import concurrent.futures
import time
from datetime import datetime

In [0]:
def do_thing (num, msg = None):

    if msg == 'Break':
        raise Exception(f'You asked to Break (num = {num})')
    if num < 0:
        raise ValueError(f'Invalid value {num} - must provide a positive number')

    # Start timing the operation
    t0 = time.time()

    # Build return string
    ret = f'{num:3}^2 = {num**2:5}'

    # Sleep a variable number (0-6) seconds
    time.sleep(num**2 % 7 + num / 7)

    # Get lap time and return in string
    t = time.time()
    ret += f'{"":4}({t - t0:2.3f} s) '

    # If optional message given, add on to end
    if msg is not None:
        ret += msg

    return ret

In [0]:
def callback (fut: concurrent.futures.Future):
    '''
    Example function to be called when the callable of a Future completes

    fut: - the Future executed to which this callback function was attached
    '''

    # Determine if the future was cancelled, failed, or succeeded
    # Cancelled
    if fut.cancelled():
        print(f'MSG: {fut} was cancelled...')
    
    # Exception encountered
    elif fut.exception():
        err = fut.exception()
        print(f'EX:  {err} exception encountered in {fut}')

    # Success! Print the result
    else:
        result = fut.result()
        print(f'{result}')


In [0]:
help(callback)

In [0]:
 # Skip this step unless you really want to do it...
if 1 == 2:
    # Setup a range and start the timer
    R = range(4)
    t0 = time.time()

    # Loop over range and call function on each value
    for r in R:
        print(do_thing(r))
        
    # Get batch end time and report
    t = time.time() - t0
    print(f'*******************************\n{t - t0:2.3f} s')

In [0]:
# Go with a bigger range here as all will start near simultaneously 
# and overall finish will be about as long as longest execution
R = range(10)

t0 = time.time()

with concurrent.futures.ThreadPoolExecutor(len(R)) as executor:
    fs = {executor.submit(do_thing, r, f'Hey, there! You wanted the square of {r}.  Hope it was {r**2}...'): r for r in R}
    for f in concurrent.futures.as_completed(fs):
        s = f'{f.result()}'
        print(s)

# Batch reporting
t = time.time()
print(f'*******************************\n{t - t0:2.3f} s')

In [0]:
# Go with a bigger range here as all will start near simultaneously 
# and overall finish will be about as long as longest execution
R = range(20)

t0 = time.time()

with concurrent.futures.ThreadPoolExecutor(len(R)) as executor:
    fs = {executor.submit(do_thing, r, f'Should be {r**2}'): r for r in R if r % 5 < 2}
    fs = fs | {executor.submit(do_thing, r): r for r in R if r % 5 == 2}
    fs = fs | {executor.submit(do_thing, r, f'Cube is {r**3}'): r for r in R if r % 5 > 2}
    for fn in concurrent.futures.as_completed(fs):
        print(f'{fn.result()}')

# Batch reporting
t = time.time()
print(f'*******************************\n{t - t0:2.3f} s')

In [0]:
R = range(-1,11)

t0 = time.time()

print(f'{t0} : Starting\n')
with concurrent.futures.ThreadPoolExecutor(len(R), thread_name_prefix='acbThread_') as executor:
    fs = {executor.submit(do_thing, r, f'\tShould be {r**2}'): r for r in R if r % 5 < 2}
    fs = fs | {executor.submit(do_thing, r, 'Break'): r for r in R if r % 5 == 2}
    fs = fs | {executor.submit(do_thing, r, f'\tCube is {r**3}'): r for r in R if r % 5 > 2}
    
    for fn in fs:
        fn.add_done_callback(callback)
    
    print(f'{time.time()} : Waiting on a future to complete')
    c = concurrent.futures.wait(fs, 2.5, concurrent.futures.ALL_COMPLETED).not_done
    print(f'{time.time()} : First future is complete\n')

# Batch reporting
t = time.time()
print(f'*******************************\n{t - t0:2.3f} s')